In [0]:
"""
id: ai_branch_group
template: group
name: AI Enrichment Branch
position:
  x: 240
  y: -40
dimensions:
  width: 1620
  height: 240
description:
  text: "Visual container around the AI enrichment branch."
config: {}
input: []
"""
# group is a visual-only container; no run body needed.


---
id: pipeline_header
template: markdown
name: Customer Feedback Pipeline
position:
  x: 0
  y: -260
dimensions:
  width: 560
  height: 220
config:
  md: |
    # Customer Feedback Pipeline
    
    **Converted from `customer_feedback.yxmd`** using the `alteryx-to-vdp` skill.
    
    Combines current + historical feedback CSVs, enriches with AI (sentiment, extraction, masking),
    joins to the products catalog, aggregates per category, ranks, and writes the gold table.
    
    **Inputs**
    - `/Volumes/aldi_aus/aldi_us/test/current_feedback.csv`
    - `/Volumes/aldi_aus/aldi_us/test/historical_feedback.csv`
    - `aldi_aus.aldi_us.products`
    
    **Output**
    - `aldi_aus.aldi_us.gold_feedback_summary`
---


In [0]:
"""
id: src_current
template: source
name: src_current
position:
  x: 0
  y: 0
description:
  text: "Read current feedback CSV from the UC Volume."
previewMode: "1000"
config:
  file_source:
    path: /Volumes/aldi_aus/aldi_us/test/current_feedback.csv
    format: csv
    header: true
    inferSchema: true
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/aldi_aus/aldi_us/test/current_feedback.csv",
        "format": "csv",
        "header": true,
        "inferSchema": true
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_current.data"] = out["data"]


In [0]:
"""
id: src_historical
template: source
name: src_historical
position:
  x: 0
  y: 145
description:
  text: "Read historical feedback CSV from the UC Volume."
previewMode: "1000"
config:
  file_source:
    path: /Volumes/aldi_aus/aldi_us/test/historical_feedback.csv
    format: csv
    header: true
    inferSchema: true
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/aldi_aus/aldi_us/test/historical_feedback.csv",
        "format": "csv",
        "header": true,
        "inferSchema": true
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_historical.data"] = out["data"]


In [0]:
"""
id: src_products
template: source
name: src_products
position:
  x: 0
  y: 720
description:
  text: "Read the products catalog from Unity Catalog."
previewMode: "1000"
config:
  table_source:
    tableName: aldi_aus.aldi_us.products
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "aldi_aus.aldi_us.products"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_products.data"] = out["data"]


In [0]:
"""
id: combine_all
template: combine
name: combine_all
position:
  x: 260
  y: 70
description:
  text: "Union all feedback (current + historical), keeping duplicates."
previewMode: "1000"
config:
  operator: UNION
  quantifier: ALL
input:
  - node: src_current
    input_port: data_0
    output_port: data
  - node: src_historical
    input_port: data_1
    output_port: data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df_0 = inputs["data_0"]
    df_1 = inputs["data_1"]
    operator = config.get("operator", "UNION")
    quantifier = config.get("quantifier", "DISTINCT")

    op_map = {
        "UNION": lambda a, b: a.union(b),
        "INTERSECT": lambda a, b: a.intersectAll(b),
        "EXCEPT": lambda a, b: a.exceptAll(b),
        "MINUS": lambda a, b: a.exceptAll(b),
    }
    op_distinct_map = {
        "UNION": lambda a, b: a.union(b).distinct(),
        "INTERSECT": lambda a, b: a.intersect(b),
        "EXCEPT": lambda a, b: a.subtract(b),
        "MINUS": lambda a, b: a.subtract(b),
    }
    if quantifier == "DISTINCT":
        combine_fn = op_distinct_map.get(operator)
    else:
        combine_fn = op_map.get(operator)
    if not combine_fn:
        raise ValueError(f"Unsupported combine operator: {operator}")

    return {"combined_data": combine_fn(df_0, df_1)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "operator": "UNION",
    "quantifier": "ALL"
}
inputs = {
    "data_0": ctx["src_current.data"],
    "data_1": ctx["src_historical.data"],
}
out = run(config, inputs, spark)
ctx["combine_all.combined_data"] = out["combined_data"]


In [0]:
"""
id: filter_valid
template: filter
name: filter_valid
position:
  x: 520
  y: 70
description:
  text: "Keep only valid feedback rows."
previewMode: "1000"
config:
  condition: rating BETWEEN 1 AND 5 AND comment IS NOT NULL
input:
  - node: combine_all
    input_port: data
    output_port: combined_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")
    if not condition:
        return {"filtered_data": df}
    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "rating BETWEEN 1 AND 5 AND comment IS NOT NULL"
}
inputs = {
    "data": ctx["combine_all.combined_data"],
}
out = run(config, inputs, spark)
ctx["filter_valid.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: transform_clean
template: transform
name: transform_clean
position:
  x: 780
  y: 70
description:
  text: "Cast types, derive a rating_band column."
previewMode: "1000"
config:
  expressions:
  - CAST(customer_id AS INT) AS `customer_id`
  - CAST(product_id AS INT) AS `product_id`
  - CAST(rating AS INT) AS `rating`
  - comment
  - "TO_DATE(feedback_date, 'yyyy-MM-dd') AS `feedback_date`"
  - CASE WHEN rating >= 4 THEN 'positive' WHEN rating = 3 THEN 'neutral' ELSE 'negative' END AS `rating_band`
input:
  - node: filter_valid
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"transformed_data": df}
    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "CAST(customer_id AS INT) AS `customer_id`",
        "CAST(product_id AS INT) AS `product_id`",
        "CAST(rating AS INT) AS `rating`",
        "comment",
        "TO_DATE(feedback_date, 'yyyy-MM-dd') AS `feedback_date`",
        "CASE WHEN rating >= 4 THEN 'positive' WHEN rating = 3 THEN 'neutral' ELSE 'negative' END AS `rating_band`"
    ]
}
inputs = {
    "data": ctx["filter_valid.filtered_data"],
}
out = run(config, inputs, spark)
ctx["transform_clean.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: ai_enrich
template: ai_function
name: ai_enrich
position:
  x: 1040
  y: 70
description:
  text: "Run sentiment, structured extraction, and PII masking on the comment column."
previewMode: "1000"
config:
  expressions:
  - ai_analyze_sentiment(comment) AS `sentiment`
  - "ai_extract(comment, ARRAY('issue_type','severity')) AS `extracted_issue`"
  - "ai_mask(comment, ARRAY('EMAIL','PHONE','PERSON')) AS `comment_masked`"
input:
  - node: transform_clean
    input_port: data
    output_port: transformed_data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}
    return {"ai_data": df.selectExpr(*expressions, "*")}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_analyze_sentiment(comment) AS `sentiment`",
        "ai_extract(comment, ARRAY('issue_type','severity')) AS `extracted_issue`",
        "ai_mask(comment, ARRAY('EMAIL','PHONE','PERSON')) AS `comment_masked`"
    ]
}
inputs = {
    "data": ctx["transform_clean.transformed_data"],
}
out = run(config, inputs, spark)
ctx["ai_enrich.ai_data"] = out["ai_data"]


In [0]:
"""
id: py_score
template: python
name: py_score
position:
  x: 1300
  y: 70
description:
  text: "Compute a weighted feedback score from rating + sentiment."
previewMode: "1000"
config:
  code: |
    from pyspark.sql.functions import col, when, lit
    df = inputs["data"][0]
    sentiment_weight = when(col("sentiment") == "positive", lit(1.0)) \
                      .when(col("sentiment") == "negative", lit(-1.0)) \
                      .otherwise(lit(0.0))
    result = df.withColumn("weighted_score", col("rating") * (lit(1.0) + sentiment_weight))
input:
  - node: ai_enrich
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    code = config.get("code", "")
    local_vars = {"inputs": inputs, "spark": spark, "result": None}
    exec(code, {}, local_vars)
    return {"result": local_vars.get("result")}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "code": "from pyspark.sql.functions import col, when, lit\ndf = inputs[\"data\"][0]\nsentiment_weight = when(col(\"sentiment\") == \"positive\", lit(1.0)) \\\n                  .when(col(\"sentiment\") == \"negative\", lit(-1.0)) \\\n                  .otherwise(lit(0.0))\nresult = df.withColumn(\"weighted_score\", col(\"rating\") * (lit(1.0) + sentiment_weight))\n"
}
inputs = {
    "data": [
        ctx["ai_enrich.ai_data"],
    ],
}
out = run(config, inputs, spark)
ctx["py_score.result"] = out["result"]


In [0]:
"""
id: sql_dedupe
template: sql
name: sql_dedupe
position:
  x: 1560
  y: 70
description:
  text: "Deduplicate to the latest feedback per (customer, product)."
previewMode: "1000"
config:
  query: |
    SELECT * EXCEPT (_dedup_rn) FROM (
      SELECT *,
             ROW_NUMBER() OVER (PARTITION BY customer_id, product_id ORDER BY feedback_date DESC) AS _dedup_rn
      FROM py_score
    ) WHERE _dedup_rn = 1
input:
  - node: py_score
    input_port: data
    output_port: result
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT * EXCEPT (_dedup_rn) FROM (\n  SELECT *,\n         ROW_NUMBER() OVER (PARTITION BY customer_id, product_id ORDER BY feedback_date DESC) AS _dedup_rn\n  FROM py_score\n) WHERE _dedup_rn = 1\n"
}
inputs = {
    "data": [
        ctx["py_score.result"],
    ],
    "data__sources": [
        {"node": "py_score", "output_port": "result", "name": "py_score", "df_name": "py_score"},
    ],
}
out = run(config, inputs, spark)
ctx["sql_dedupe.result"] = out["result"]


In [0]:
"""
id: join_products
template: join
name: join_products
position:
  x: 1820
  y: 300
description:
  text: "Left join the deduped feedback with the products catalog."
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.product_id = right.product_id
  expressions: []
input:
  - node: sql_dedupe
    input_port: left
    output_port: result
  - node: src_products
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)
    if expressions:
        result = result.selectExpr(*expressions)
    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.product_id = right.product_id",
    "expressions": []
}
inputs = {
    "left": ctx["sql_dedupe.result"],
    "right": ctx["src_products.data"],
}
out = run(config, inputs, spark)
ctx["join_products.joined_data"] = out["joined_data"]


In [0]:
"""
id: aggregate_by_category
template: aggregate
name: aggregate_by_category
position:
  x: 2080
  y: 300
description:
  text: "Per-category metrics: avg / median / sum / count."
previewMode: "1000"
config:
  group_bys:
  - expr: category
    type: expr
  aggregations:
  - columnExpr:
        expr: weighted_score
        type: expr
    fn: AVG
    alias: avg_weighted_score
  - columnExpr:
        expr: rating
        type: expr
    fn: AVG
    alias: avg_rating
  - columnExpr:
        expr: rating
        type: expr
    fn: COUNT
    alias: feedback_count
  - columnExpr:
        expr: weighted_score
        type: expr
    fn: MEDIAN
    alias: median_weighted_score
  - columnExpr:
        expr: weighted_score
        type: expr
    fn: SUM
    alias: sum_weighted_score
input:
  - node: join_products
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum, "AVG": F.avg, "COUNT": F.count, "MIN": F.min, "MAX": F.max,
            "MEAN": F.mean, "MEDIAN": F.median, "STDDEV": F.stddev, "VARIANCE": F.variance,
        }
        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.col(raw_expr)
        else:
            col = F.expr(f"{fn}({raw_expr})")
        if alias:
            col = col.alias(alias)
        agg_exprs.append(col)

    group_cols = [gb.get("expr", "") for gb in group_bys if gb.get("expr", "")]
    if not agg_exprs:
        if group_cols:
            return {"aggregated_data": df.select(*group_cols).distinct()}
        return {"aggregated_data": df}
    if group_cols:
        return {"aggregated_data": df.groupBy(*group_cols).agg(*agg_exprs)}
    return {"aggregated_data": df.agg(*agg_exprs)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "category",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "weighted_score",
                "type": "expr"
            },
            "fn": "AVG",
            "alias": "avg_weighted_score"
        },
        {
            "columnExpr": {
                "expr": "rating",
                "type": "expr"
            },
            "fn": "AVG",
            "alias": "avg_rating"
        },
        {
            "columnExpr": {
                "expr": "rating",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "feedback_count"
        },
        {
            "columnExpr": {
                "expr": "weighted_score",
                "type": "expr"
            },
            "fn": "MEDIAN",
            "alias": "median_weighted_score"
        },
        {
            "columnExpr": {
                "expr": "weighted_score",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "sum_weighted_score"
        }
    ]
}
inputs = {
    "data": ctx["join_products.joined_data"],
}
out = run(config, inputs, spark)
ctx["aggregate_by_category.aggregated_data"] = out["aggregated_data"]


In [0]:
"""
id: pivot_rating_dist
template: pivot
name: pivot_rating_dist
position:
  x: 2080
  y: 460
description:
  text: "Distribution of ratings 1..5 per category."
previewMode: "1000"
config:
  mode: pivot
  group_by:
  - category
  pivot_column: rating
  value_column: rating
  agg: count
input:
  - node: join_products
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    mode = config.get("mode", "pivot")
    if mode == "pivot":
        group_by = config.get("group_by", [])
        pivot_col = config.get("pivot_column")
        value_col = config.get("value_column")
        agg_fn = config.get("agg", "count").lower()
        result = df.groupBy(*group_by).pivot(pivot_col).agg(getattr(F, agg_fn)(value_col))
        return {"pivoted_data": result}
    else:
        id_cols: List[str] = config.get("id_columns", [])
        value_cols: List[str] = config.get("value_columns", [])
        key_name = config.get("key_name", "key")
        value_name = config.get("value_name", "value")
        stack_expr = "stack({n}, {pairs}) AS ({k}, {v})".format(
            n=len(value_cols),
            pairs=", ".join(f"'{c}', `{c}`" for c in value_cols),
            k=key_name,
            v=value_name,
        )
        result = df.selectExpr(*id_cols, stack_expr)
        return {"pivoted_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "pivot",
    "group_by": [
        "category"
    ],
    "pivot_column": "rating",
    "value_column": "rating",
    "agg": "count"
}
inputs = {
    "data": ctx["join_products.joined_data"],
}
out = run(config, inputs, spark)
ctx["pivot_rating_dist.pivoted_data"] = out["pivoted_data"]


In [0]:
"""
id: join_pivot
template: join
name: join_pivot
position:
  x: 2340
  y: 380
description:
  text: "Combine aggregates with the pivoted distribution."
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.category = right.category
  expressions: []
input:
  - node: aggregate_by_category
    input_port: left
    output_port: aggregated_data
  - node: pivot_rating_dist
    input_port: right
    output_port: pivoted_data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)
    if expressions:
        result = result.selectExpr(*expressions)
    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.category = right.category",
    "expressions": []
}
inputs = {
    "left": ctx["aggregate_by_category.aggregated_data"],
    "right": ctx["pivot_rating_dist.pivoted_data"],
}
out = run(config, inputs, spark)
ctx["join_pivot.joined_data"] = out["joined_data"]


In [0]:
"""
id: sort_by_score
template: sort
name: sort_by_score
position:
  x: 2600
  y: 380
description:
  text: "Sort categories by average weighted score, descending."
previewMode: "1000"
config:
  sort_expressions:
  - columnExpr:
        expr: avg_weighted_score
        type: expr
    sortBy: DESC
input:
  - node: join_pivot
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])
    if not sort_expressions:
        return {"sorted_data": df}
    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")
        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()
        order_cols.append(col)
    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "avg_weighted_score",
                "type": "expr"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["join_pivot.joined_data"],
}
out = run(config, inputs, spark)
ctx["sort_by_score.sorted_data"] = out["sorted_data"]


In [0]:
"""
id: limit_top_n
template: limit
name: limit_top_n
position:
  x: 2860
  y: 380
description:
  text: "Keep the top 10 categories."
previewMode: "1000"
config:
  n: 10
input:
  - node: sort_by_score
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    n = int(config.get("n", 100))
    return {"limited_data": df.limit(n)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "n": 10
}
inputs = {
    "data": ctx["sort_by_score.sorted_data"],
}
out = run(config, inputs, spark)
ctx["limit_top_n.limited_data"] = out["limited_data"]


In [0]:
"""
id: output_gold
template: output
name: output_gold
position:
  x: 3120
  y: 380
description:
  text: "Materialize the result as a Unity Catalog Delta table."
previewMode: "1000"
config:
  catalog: aldi_aus
  schema: aldi_us
  table_name: gold_feedback_summary
input:
  - node: limit_top_n
    input_port: data
    output_port: limited_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    if not table_name:
        raise ValueError("Output: 'table_name' is required")
    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)
    df.write.mode("overwrite").saveAsTable(full_name)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "aldi_aus",
    "schema": "aldi_us",
    "table_name": "gold_feedback_summary"
}
inputs = {
    "data": ctx["limit_top_n.limited_data"],
}
out = run(config, inputs, spark)
